In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from matplotlib import rcParams

rcParams['figure.figsize'] = 12, 4
rcParams['lines.linewidth'] = 3
rcParams['xtick.labelsize'] = 'x-large'
rcParams['ytick.labelsize'] = 'x-large'

In [ ]:
from google.colab import files

fileupload = files.upload()

In [ ]:
df = pd.read_csv('train.csv', sep=';')

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Filtering data untuk poutcome='success'
success_df = df[df['poutcome'] == 'success']

# Menampilkan 5 data pertama dari DataFrame yang telah difilter
print(success_df.head())

# EDA (Exploratory Data Analysis)

success_df.describe()
# 1. Univariate Analysis
# -------------------------------
# Histogram untuk variabel numerik (misalnya, 'age', 'balance', 'duration')
import matplotlib.pyplot as plt
import seaborn as sns

numeric_columns = ['age', 'balance', 'duration']
for column in numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(success_df[column], kde=True)
    plt.title(f'Histogram of {column}')
    plt.show()

# Countplot untuk variabel kategorikal (misalnya, 'job', 'marital', 'education')
categorical_columns = ['job', 'marital', 'education']
for column in categorical_columns:
    plt.figure(figsize=(10, 5))
    sns.countplot(y=success_df[column])
    plt.title(f'Countplot of {column}')
    plt.show()

# -------------------------------
# 2. Multivariate Analysis
# -------------------------------
# Scatter plot untuk melihat hubungan antara variabel numerik
plt.figure(figsize=(10, 6))
sns.scatterplot(x='balance', y='duration', data=success_df, hue='y')
plt.title('Scatter Plot of Balance vs Duration')
plt.show()

# Pair plot untuk melihat hubungan antara beberapa variabel numerik
sns.pairplot(success_df[['age', 'balance', 'duration', 'y']], hue='y')
plt.suptitle('Pair Plot of Age, Balance, and Duration')
plt.show()

#Korelasi Numerical To Categorical

In [ ]:
# pengelompokan kolom berdasarkan jenisnya
cats = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact','poutcome','y']
nums = ['age', 'balance', 'campaign', 'pdays', 'previous']
timestamp = ['day', 'month', 'duration']

In [ ]:
from scipy.stats import f_oneway

# Pisahkan data berdasarkan kategori target
for feature in ['age', 'balance', 'campaign', 'pdays', 'previous']:
    grouped_data = [df[feature][df['y'] == category] for category in df['y'].unique()]

    # Lakukan uji ANOVA
    anova_result = f_oneway(*grouped_data)

    # Tampilkan hasil uji ANOVA
    print(f"Hasil Uji ANOVA untuk {feature}:")
    print("Statistik Uji F:", anova_result.statistic)
    print("p-value:", anova_result.pvalue)

    # Interpretasi hasil uji ANOVA
    if anova_result.pvalue < 0.05:
        print(f"Terdapat perbedaan signifikan dalam rata-rata {feature} antara kelompok target.")
    else:
        print(f"Tidak terdapat perbedaan signifikan dalam rata-rata {feature} antara kelompok target.")


Feature Selection:

Jika uji ANOVA menunjukkan perbedaan signifikan untuk suatu fitur, Anda dapat mempertimbangkan fitur tersebut untuk dimasukkan ke dalam analisis atau model lebih lanjut.

# Handle missing values

In [ ]:
df.isna().sum()

Tidak terdapat missing values pada dataset

In [ ]:
missing_data = df[(df['poutcome'] == 'unknown') & (df['pdays'] != -1)]

missing_data

In [ ]:
mask = ~((df['poutcome'] == 'unknown') & (df['pdays'] != -1))

df = df[mask]

Hapus data unknown yang missing values dari poutcome dan tinggalkan unknown yang client belum pernah dikontak

In [ ]:
df['poutcome'].replace({'unknown': 'never'}, inplace=True)

### Mengganti nilai unknown pada fitur contact dengan modus

In [ ]:
# Mengganti nilai unknown dengan modus
modus_contact = df['contact'].mode()[0]
df['contact'] = df['contact'].replace('unknown', modus_contact)

In [ ]:
df['contact'].unique()

In [ ]:
df.head()

In [ ]:
for col in cats:
    print(f'''Value count kolom {col}:''')
    print(df[col].value_counts())
    print()

# Handle duplicated data

In [ ]:
df.duplicated().sum()

Tidak terdapat duplicated values pada dataset

# Handle outliers

In [ ]:
nums2 = ['age','campaign']
for num in nums2:
  df[num] = np.log(df[num])


In [ ]:
from scipy import stats

print("Before removing outlier: ", len(df))

for num in nums2:
  z_scores = np.abs(stats.zscore(df[num]))
  df = df[z_scores < 3]

print("After removing outlier: ", len(df))

In [ ]:
for i in range(0, len(nums)):
    plt.subplot(1, len(nums), i+1)
    sns.boxplot(y=df[nums[i]], color='orange', orient='v')
    plt.tight_layout()

# Feature transformation

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = df.drop(['y'], axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

print(f'Number of Train Data: {y_train.shape[0]}')
print(f'Number of Test Data: {y_test.shape[0]}')


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
columns_to_standardize = ['age', 'balance', 'campaign', 'pdays', 'previous']
X_train[columns_to_standardize] = scaler.fit_transform(X_train[columns_to_standardize])
X_test[columns_to_standardize] = scaler.transform(X_test[columns_to_standardize])
print("DataFrame setelah distandardisasi:")
X_train.head()

In [ ]:
X_test.head()

In [ ]:
df.info()

# Feature encoding

cats = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact','poutcome','y']

In [ ]:
mapping_default = {
    'no' : 0,
    'yes' : 1,
    }
X_train['default'] = X_train['default'].map(mapping_default)
X_test['default'] = X_test['default'].map(mapping_default)

In [ ]:
mapping_housing = {
    'no' : 0,
    'yes' : 1,
    }
X_train['housing'] = X_train['housing'].map(mapping_housing)
X_test['housing'] = X_test['housing'].map(mapping_housing)

In [ ]:
mapping_loan = {
    'no' : 0,
    'yes' : 1,
    }
X_train['loan'] = X_train['loan'].map(mapping_loan)
X_test['loan'] = X_test['loan'].map(mapping_loan)

In [ ]:
X_train_encoded_education = pd.get_dummies(X_train['education'], prefix = 'pendidikan')
X_test_encoded_education = pd.get_dummies(X_test['education'], prefix = 'pendidikan')

In [ ]:
X_train_encoded_kerja = pd.get_dummies(X_train['job'], prefix = 'kerja')
X_test_encoded_kerja = pd.get_dummies(X_test['job'], prefix = 'kerja')

In [ ]:
X_train_encoded_marital = pd.get_dummies(X_train['marital'], prefix = 'status')
X_test_encoded_marital = pd.get_dummies(X_test['marital'], prefix = 'status')

In [ ]:
X_train_encoded_contact = pd.get_dummies(X_train['contact'], prefix = 'contact')
X_test_encoded_contact = pd.get_dummies(X_test['contact'], prefix = 'contact')

In [ ]:
X_train_encoded_poutcome = pd.get_dummies(X_train['poutcome'], prefix = 'poutcome')
X_test_encoded_poutcome = pd.get_dummies(X_test['poutcome'], prefix = 'poutcome')

In [ ]:
X_train.head()

In [ ]:
X_train_combined = pd.concat([X_train, X_train_encoded_education, X_train_encoded_kerja,X_train_encoded_marital,X_train_encoded_contact, X_train_encoded_poutcome], axis=1)
X_test_combined = pd.concat([X_test, X_test_encoded_education, X_test_encoded_kerja,X_test_encoded_marital,X_test_encoded_contact, X_test_encoded_poutcome], axis=1)

In [ ]:
X_train_combined = X_train_combined.drop(['job','education', 'marital','contact','month','poutcome'], axis=1)

In [ ]:
X_test_combined = X_test_combined.drop(['job','education','marital','contact','month','poutcome'], axis=1)

In [ ]:
X_train_combined.head()

In [ ]:
X_train_combined.columns

In [ ]:
X_train_combined.info()

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(X_train_combined[nums].corr(), cmap='Blues', annot=True, fmt='.2f')

In [ ]:
cats2 = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact','poutcome']

In [ ]:
plt.figure(figsize=(25,20))
sns.heatmap(X_train_combined.corr(method='kendall'), cmap='Blues', annot=True, fmt='.2f')

# Handle class imbalance

In [ ]:
y_train.value_counts()

In [ ]:
# OVERSAMPLING
from imblearn import over_sampling
X_oversampling , y_oversampling = over_sampling.SMOTE(random_state=42).fit_resample(X_train_combined,y_train)
print(pd.Series(y_oversampling).value_counts())

 Modeling

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import numpy as np

# Inisialisasi model
logreg_model = LogisticRegression(max_iter=1000)  # Menambahkan max_iter untuk mengatasi konvergensi
dt_model = DecisionTreeClassifier()
rf_model = RandomForestClassifier()
adaboost_model = AdaBoostClassifier()

# Model Dictionary
models = {
    'Logistic Regression': logreg_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'Adaboost': adaboost_model
}

# Evaluasi setiap model
for model_name, model in models.items():
    # Training model
    model.fit(X_train_combined, y_train)

    # Prediksi pada data test
    y_pred_test = model.predict(X_test_combined)
    y_pred_train = model.predict(X_train_combined)

    # AUC Score untuk data test (jika model mendukung predict_proba)
    if hasattr(model, 'predict_proba'):
        y_pred_proba_test = model.predict_proba(X_test_combined)[:, 1]
        auc_test = roc_auc_score(y_test, y_pred_proba_test)
    else:
        auc_test = None

    # AUC Score untuk data train (jika model mendukung predict_proba)
    if hasattr(model, 'predict_proba'):
        y_pred_proba_train = model.predict_proba(X_train_combined)[:, 1]
        auc_train = roc_auc_score(y_train, y_pred_proba_train)
    else:
        auc_train = None

    # Evaluasi metrik pada data test
    accuracy_test = accuracy_score(y_test, y_pred_test)
    classification_report_test = classification_report(y_test, y_pred_test)

    # Evaluasi metrik pada data train
    accuracy_train = accuracy_score(y_train, y_pred_train)
    classification_report_train = classification_report(y_train, y_pred_train)

    # Menampilkan hasil evaluasi
    print(f'{model_name} Evaluation:\n{"-"*50}\n')
    print(f'Train Accuracy: {accuracy_train:.2f}\nTrain Classification Report:\n{classification_report_train}\n')
    print(f'Test Accuracy: {accuracy_test:.2f}\nTest Classification Report:\n{classification_report_test}\n')
    if auc_train is not None:
        print(f'Train AUC Score: {auc_train:.2f}\nTest AUC Score: {auc_test:.2f}\n{"="*70}\n')
    else:
        print('Model does not support predict_proba.\n{"="*70}\n')

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Inisialisasi model XGBoost dengan konversi nilai kelas
xgb_model = xgb.XGBClassifier(objective='binary:logistic')
y_train_binary_xgb = y_train.replace({'yes': 1, 'no': 0})
xgb_model.fit(X_train_combined, y_train_binary_xgb)

# Prediksi pada data test dan train
y_pred_test_xgb = xgb_model.predict(X_test_combined)
y_pred_train_xgb = xgb_model.predict(X_train_combined)

# Menentukan variabel y_train_binary sesuai dengan label biner yang diharapkan
y_test_binary_xgb = y_test.replace({'yes': 1, 'no': 0})
y_train_binary = y_train.replace({'yes': 1, 'no': 0})

# Mengecek apakah model mendukung predict_proba
if hasattr(xgb_model, 'predict_proba'):
    y_pred_proba_test_xgb = xgb_model.predict_proba(X_test_combined)[:, 1]
    auc_test_xgb = roc_auc_score(y_test_binary_xgb, y_pred_proba_test_xgb)
else:
    auc_test_xgb = None

# AUC Score untuk data train
y_pred_proba_train_xgb = xgb_model.predict_proba(X_train_combined)[:, 1]
auc_train_xgb = roc_auc_score(y_train_binary, y_pred_proba_train_xgb)

# Evaluasi metrik pada data test
accuracy_test_xgb = accuracy_score(y_test_binary_xgb, y_pred_test_xgb)
classification_report_test_xgb = classification_report(y_test_binary_xgb, y_pred_test_xgb)

# Evaluasi metrik pada data train
accuracy_train_xgb = accuracy_score(y_train_binary_xgb, y_pred_train_xgb)
classification_report_train_xgb = classification_report(y_train_binary_xgb, y_pred_train_xgb)

# Menampilkan hasil evaluasi
print('XGBoost Evaluation:\n{"-"*50}\n')
print(f'Train Accuracy: {accuracy_train_xgb:.2f}\nTrain Classification Report:\n{classification_report_train_xgb}\n')
print(f'Test Accuracy: {accuracy_test_xgb:.2f}\nTest Classification Report:\n{classification_report_test_xgb}\n')
print(f'Train AUC Score: {auc_train_xgb:.2f}\nTest AUC Score: {auc_test_xgb:.2f}\n{"="*70}\n')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# B. Modeling:
model = RandomForestClassifier()
model.fit(X_train_combined, y_train)

In [ ]:
# C. Model Evaluation:
y_pred_test = model.predict(X_test_combined)

# Evaluasi metrik pada data test
accuracy_test = accuracy_score(y_test, y_pred_test)
classification_report_test = classification_report(y_test, y_pred_test)

print(f'Model Evaluation on Test Data:\nAccuracy: {accuracy_test:.2f}\nClassification Report:\n{classification_report_test}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Inisialisasi model Logistic Regression dengan penyesuaian
logreg_model = LogisticRegression(solver='liblinear', max_iter=1000)
dt_model = DecisionTreeClassifier()
rf_model = RandomForestClassifier()
adaboost_model = AdaBoostClassifier()

# Model Dictionary
models = {
    'Logistic Regression': logreg_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'Adaboost': adaboost_model
}

# Scaling data
scaler = StandardScaler()
X_train_combined_scaled = scaler.fit_transform(X_train_combined)
X_test_combined_scaled = scaler.transform(X_test_combined)

# Menentukan variabel y_train_binary_models sesuai dengan label biner yang diharapkan
y_train_binary_models = y_train.replace({'yes': 1, 'no': 0})

# Cross-Validation untuk model lainnya
for model_name, model in models.items():
    cross_val_scores = cross_val_score(model, X_train_combined_scaled, y_train_binary_models, cv=5)

    print(f'{model_name} Cross-Validation Scores: {cross_val_scores}')
    print(f'{model_name} Mean Cross-Validation Accuracy: {cross_val_scores.mean():.2f}\n{"-"*50}')

# Cross-Validation untuk XGBoost
xgb_cross_val_scores = cross_val_score(xgb_model, X_train_combined_scaled, y_train_binary_xgb, cv=5)

print(f'XGBoost Cross-Validation Scores: {xgb_cross_val_scores}')
print(f'XGBoost Mean Cross-Validation Accuracy: {xgb_cross_val_scores.mean():.2f}\n{"="*70}\n')

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Inisialisasi model Logistic Regression dengan penyesuaian
logreg_model = LogisticRegression(solver='liblinear', max_iter=1000)
dt_model = DecisionTreeClassifier()
rf_model = RandomForestClassifier()
adaboost_model = AdaBoostClassifier()

# Model Dictionary
models = {
    'Logistic Regression': logreg_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'Adaboost': adaboost_model
}

# Scaling data
scaler = StandardScaler()
X_train_combined_scaled = scaler.fit_transform(X_train_combined)
X_test_combined_scaled = scaler.transform(X_test_combined)

# Konversi label menjadi numerik untuk Logistic Regression
label_encoder = LabelEncoder()
y_train_binary_models = label_encoder.fit_transform(y_train)
y_test_binary_models = label_encoder.transform(y_test)

# Hyperparameter Tuning untuk model lainnya
for model_name, model in models.items():
    param_grid = {}  # Definisikan parameter grid sesuai dengan model

    grid_search = GridSearchCV(model, param_grid, cv=5)
    grid_search.fit(X_train_combined_scaled, y_train_binary_models)

    best_params = grid_search.best_params_

    print(f'Best Hyperparameters for {model_name}: {best_params}')

    # Melatih model dengan hyperparameter terbaik
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_combined_scaled, y_train_binary_models)

    # Evaluasi model setelah hyperparameter tuning pada data test
    y_pred_test_tuned = best_model.predict(X_test_combined_scaled)
    accuracy_test_tuned = accuracy_score(y_test_binary_models, y_pred_test_tuned)
    classification_report_test_tuned = classification_report(y_test_binary_models, y_pred_test_tuned)

    print(f'{model_name} Evaluation on Test Data after Hyperparameter Tuning:\nAccuracy: {accuracy_test_tuned:.2f}\nClassification Report:\n{classification_report_test_tuned}\n{"-"*50}')

In [ ]:
# Hyperparameter Tuning untuk XGBoost
param_grid_xgb = {'n_estimators': [50, 100, 200], 'max_depth': [3, 6, 9]}
grid_search_xgb = GridSearchCV(xgb.XGBClassifier(), param_grid_xgb, cv=5)
grid_search_xgb.fit(X_train_combined_scaled, y_train_binary_xgb)

best_params_xgb = grid_search_xgb.best_params_

print(f'Best Hyperparameters for XGBoost: {best_params_xgb}')

# Melatih model XGBoost dengan hyperparameter terbaik
best_model_xgb = grid_search_xgb.best_estimator_
best_model_xgb.fit(X_train_combined_scaled, y_train_binary_xgb)

# Convert true labels to numeric
y_test_numeric = y_test.replace({'no': 0, 'yes': 1})

# Evaluasi model XGBoost setelah hyperparameter tuning pada data test
y_pred_test_tuned_xgb = best_model_xgb.predict(X_test_combined_scaled)

# Menampilkan hasil evaluasi model XGBoost setelah hyperparameter tuning pada data test
print(f'XGBoost Evaluation on Test Data after Hyperparameter Tuning:\n')
print(f'Accuracy: {accuracy_score(y_test_numeric, y_pred_test_tuned_xgb):.2f}')
print(f'Classification Report:\n{classification_report(y_test_numeric, y_pred_test_tuned_xgb)}')

In [ ]:
from sklearn.metrics import roc_auc_score

# AUC Score untuk data test setelah hyperparameter tuning
y_pred_proba_test_tuned_xgb = best_model_xgb.predict_proba(X_test_combined_scaled)[:, 1]
auc_test_tuned_xgb = roc_auc_score(y_test_numeric, y_pred_proba_test_tuned_xgb)

# AUC Score untuk data train setelah hyperparameter tuning
y_pred_proba_train_tuned_xgb = best_model_xgb.predict_proba(X_train_combined_scaled)[:, 1]
auc_train_tuned_xgb = roc_auc_score(y_train_binary_xgb, y_pred_proba_train_tuned_xgb)

# Menampilkan hasil evaluasi AUC Score
print(f'AUC Score for XGBoost on Test Data after Hyperparameter Tuning: {auc_test_tuned_xgb:.2f}')
print(f'AUC Score for XGBoost on Train Data after Hyperparameter Tuning: {auc_train_tuned_xgb:.2f}\n')

In [ ]:
# Sebelum Hyperparameter Tuning - Logistic Regression
from sklearn.linear_model import LogisticRegression

# Convert true labels to numeric
y_train_binary_lr = y_train.replace({'no': 0, 'yes': 1})

# Hyperparameter Tuning untuk Logistic Regression
param_grid_lr = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
grid_search_lr = GridSearchCV(LogisticRegression(), param_grid_lr, cv=5)
grid_search_lr.fit(X_train_combined_scaled, y_train_binary_lr)

best_params_lr = grid_search_lr.best_params_

print(f'Best Hyperparameters for Logistic Regression: {best_params_lr}')

# Melatih model Logistic Regression dengan hyperparameter terbaik
best_model_lr = grid_search_lr.best_estimator_
best_model_lr.fit(X_train_combined_scaled, y_train_binary_lr)

# Evaluasi model Logistic Regression setelah hyperparameter tuning pada data test
y_pred_test_tuned_lr = best_model_lr.predict(X_test_combined_scaled)

# Menampilkan hasil evaluasi model Logistic Regression setelah hyperparameter tuning pada data test
print(f'Logistic Regression Evaluation on Test Data after Hyperparameter Tuning:\n')
print(f'Accuracy: {accuracy_score(y_test_numeric, y_pred_test_tuned_lr):.2f}')
print(f'Classification Report:\n{classification_report(y_test_numeric, y_pred_test_tuned_lr)}')

# AUC Score untuk data test setelah hyperparameter tuning
y_pred_proba_test_tuned_lr = best_model_lr.predict_proba(X_test_combined_scaled)[:, 1]
auc_test_tuned_lr = roc_auc_score(y_test_numeric, y_pred_proba_test_tuned_lr)

# AUC Score untuk data train setelah hyperparameter tuning
y_pred_proba_train_tuned_lr = best_model_lr.predict_proba(X_train_combined_scaled)[:, 1]
auc_train_tuned_lr = roc_auc_score(y_train_binary_lr, y_pred_proba_train_tuned_lr)

# Menampilkan hasil evaluasi AUC Score
print(f'AUC Score for Logistic Regression on Test Data after Hyperparameter Tuning: {auc_test_tuned_lr:.2f}')
print(f'AUC Score for Logistic Regression on Train Data after Hyperparameter Tuning: {auc_train_tuned_lr:.2f}\n')

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Convert true labels to numeric
y_train_binary_dt = y_train.replace({'no': 0, 'yes': 1})

# Hyperparameter Tuning untuk Decision Tree
param_grid_dt = {'max_depth': [3, 6, 9, None], 'min_samples_split': [2, 5, 10]}
grid_search_dt = GridSearchCV(DecisionTreeClassifier(), param_grid_dt, cv=5)
grid_search_dt.fit(X_train_combined_scaled, y_train_binary_dt)

best_params_dt = grid_search_dt.best_params_

print(f'Best Hyperparameters for Decision Tree: {best_params_dt}')

# Melatih model Decision Tree dengan hyperparameter terbaik
best_model_dt = grid_search_dt.best_estimator_
best_model_dt.fit(X_train_combined_scaled, y_train_binary_dt)

# Evaluasi model Decision Tree setelah hyperparameter tuning pada data test
y_pred_test_tuned_dt = best_model_dt.predict(X_test_combined_scaled)

# Menampilkan hasil evaluasi model Decision Tree setelah hyperparameter tuning pada data test
print(f'Decision Tree Evaluation on Test Data after Hyperparameter Tuning:\n')
print(f'Accuracy: {accuracy_score(y_test_numeric, y_pred_test_tuned_dt):.2f}')
print(f'Classification Report:\n{classification_report(y_test_numeric, y_pred_test_tuned_dt)}')

# AUC Score untuk data test setelah hyperparameter tuning
y_pred_proba_test_tuned_dt = best_model_dt.predict_proba(X_test_combined_scaled)[:, 1]
auc_test_tuned_dt = roc_auc_score(y_test_numeric, y_pred_proba_test_tuned_dt)

# AUC Score untuk data train setelah hyperparameter tuning
y_pred_proba_train_tuned_dt = best_model_dt.predict_proba(X_train_combined_scaled)[:, 1]
auc_train_tuned_dt = roc_auc_score(y_train_binary_dt, y_pred_proba_train_tuned_dt)

# Menampilkan hasil evaluasi AUC Score
print(f'AUC Score for Decision Tree on Test Data after Hyperparameter Tuning: {auc_test_tuned_dt:.2f}')
print(f'AUC Score for Decision Tree on Train Data after Hyperparameter Tuning: {auc_train_tuned_dt:.2f}\n')

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Convert true labels to numeric
y_train_binary_rf = y_train.replace({'no': 0, 'yes': 1})

# Hyperparameter Tuning untuk Random Forest
param_grid_rf = {'n_estimators': [50, 100, 200], 'max_depth': [3, 6, 9], 'min_samples_split': [2, 5, 10]}
grid_search_rf = GridSearchCV(RandomForestClassifier(), param_grid_rf, cv=5)
grid_search_rf.fit(X_train_combined_scaled, y_train_binary_rf)

best_params_rf = grid_search_rf.best_params_

print(f'Best Hyperparameters for Random Forest: {best_params_rf}')

# Melatih model Random Forest dengan hyperparameter terbaik
best_model_rf = grid_search_rf.best_estimator_
best_model_rf.fit(X_train_combined_scaled, y_train_binary_rf)

# Evaluasi model Random Forest setelah hyperparameter tuning pada data test
y_pred_test_tuned_rf = best_model_rf.predict(X_test_combined_scaled)

# Menampilkan hasil evaluasi model Random Forest setelah hyperparameter tuning pada data test
print(f'Random Forest Evaluation on Test Data after Hyperparameter Tuning:\n')
print(f'Accuracy: {accuracy_score(y_test_numeric, y_pred_test_tuned_rf):.2f}')
print(f'Classification Report:\n{classification_report(y_test_numeric, y_pred_test_tuned_rf)}')

# AUC Score untuk data test setelah hyperparameter tuning
y_pred_proba_test_tuned_rf = best_model_rf.predict_proba(X_test_combined_scaled)[:, 1]
auc_test_tuned_rf = roc_auc_score(y_test_numeric, y_pred_proba_test_tuned_rf)

# AUC Score untuk data train setelah hyperparameter tuning
y_pred_proba_train_tuned_rf = best_model_rf.predict_proba(X_train_combined_scaled)[:, 1]
auc_train_tuned_rf = roc_auc_score(y_train_binary_rf, y_pred_proba_train_tuned_rf)

# Menampilkan hasil evaluasi AUC Score
print(f'AUC Score for Random Forest on Test Data after Hyperparameter Tuning: {auc_test_tuned_rf:.2f}')
print(f'AUC Score for Random Forest on Train Data after Hyperparameter Tuning: {auc_train_tuned_rf:.2f}\n')

In [ ]:
# Mendapatkan feature importance dari model XGBoost terbaik
feature_importance_xgb = best_model_xgb.feature_importances_

# Membuat DataFrame untuk lebih mudah diproses
feature_importance_df_xgb = pd.DataFrame({'Feature': X_train_combined.columns, 'Importance': feature_importance_xgb})

# Mengurutkan DataFrame berdasarkan importance secara descending
feature_importance_df_xgb = feature_importance_df_xgb.sort_values(by='Importance', ascending=False)

# Menampilkan feature importance
print("Feature Importance (XGBoost):")
print(feature_importance_df_xgb)

# Visualisasi feature importance
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df_xgb)
plt.title('Feature Importance (XGBoost)')
plt.savefig('fi.png')
plt.show()

In [ ]:
# Mendapatkan feature importance dari model terbaik
feature_importance_xgb = best_model_xgb.feature_importances_

# Membuat DataFrame untuk lebih mudah diproses
feature_importance_df_xgb = pd.DataFrame({'Feature': X_train_combined.columns, 'Importance': feature_importance_xgb})

# Mengurutkan DataFrame berdasarkan importance secara descending
feature_importance_df_xgb = feature_importance_df_xgb.sort_values(by='Importance', ascending=False)

# Buang fitur dengan importance di bawah 0.025
selected_features = feature_importance_df_xgb[feature_importance_df_xgb['Importance'] >= 0.030]['Feature'].tolist()
X_train_combined_selected = X_train_combined[selected_features]
X_test_combined_selected = X_test_combined[selected_features]

# Hyperparameter Tuning untuk XGBoost setelah fitur selection
param_grid_xgb_selected = {'n_estimators': [50, 100, 200], 'max_depth': [3, 6, 9]}
grid_search_xgb_selected = GridSearchCV(xgb.XGBClassifier(), param_grid_xgb_selected, cv=5)
grid_search_xgb_selected.fit(X_train_combined_selected, y_train_binary_xgb)

best_params_xgb_selected = grid_search_xgb_selected.best_params_

print(f'Best Hyperparameters for XGBoost after Feature Selection: {best_params_xgb_selected}')

# Melatih model XGBoost dengan hyperparameter terbaik setelah fitur selection
best_model_xgb_selected = grid_search_xgb_selected.best_estimator_
best_model_xgb_selected.fit(X_train_combined_selected, y_train_binary_xgb)

# Evaluasi model XGBoost setelah hyperparameter tuning dan fitur selection
y_pred_train_xgb_selected = best_model_xgb_selected.predict(X_train_combined_selected)
y_pred_test_xgb_selected = best_model_xgb_selected.predict(X_test_combined_selected)

# AUC Score untuk data train setelah fitur selection
y_pred_proba_train_xgb_selected = best_model_xgb_selected.predict_proba(X_train_combined_selected)[:, 1]
auc_train_xgb_selected = roc_auc_score(y_train_binary_xgb, y_pred_proba_train_xgb_selected)

# AUC Score untuk data test setelah fitur selection
y_pred_proba_test_xgb_selected = best_model_xgb_selected.predict_proba(X_test_combined_selected)[:, 1]
auc_test_xgb_selected = roc_auc_score(y_test_binary_xgb, y_pred_proba_test_xgb_selected)

# Evaluasi metrik pada data train setelah fitur selection
accuracy_train_xgb_selected = accuracy_score(y_train_binary_xgb, y_pred_train_xgb_selected)
classification_report_train_xgb_selected = classification_report(y_train_binary_xgb, y_pred_train_xgb_selected)

# Evaluasi metrik pada data test setelah fitur selection
accuracy_test_xgb_selected = accuracy_score(y_test_binary_xgb, y_pred_test_xgb_selected)
classification_report_test_xgb_selected = classification_report(y_test_binary_xgb, y_pred_test_xgb_selected)

# Menampilkan hasil evaluasi setelah fitur selection
print('XGBoost Evaluation after Feature Selection:\n' + "-"*50 + '\n')
print(f'Train Accuracy: {accuracy_train_xgb_selected:.2f}\nTrain Classification Report:\n{classification_report_train_xgb_selected}\n')
print(f'Test Accuracy: {accuracy_test_xgb_selected:.2f}\nTest Classification Report:\n{classification_report_test_xgb_selected}\n')
print(f'Train AUC Score: {auc_train_xgb_selected:.2f}\nTest AUC Score: {auc_test_xgb_selected:.2f}\n' + "="*70 + '\n')

In [ ]:
# Evaluasi model XGBoost sebelum hyperparameter tuning pada data test
y_pred_test_xgb = best_model_xgb.predict(X_test_combined_scaled)

# Menampilkan hasil evaluasi model XGBoost sebelum hyperparameter tuning pada data test
print(f'XGBoost Evaluation on Test Data before Hyperparameter Tuning:\n')
print(f'Accuracy: {accuracy_score(y_test_numeric, y_pred_test_xgb):.2f}')
print(f'Classification Report:\n{classification_report(y_test_numeric, y_pred_test_xgb)}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Membuat DataFrame untuk feature importance setelah pemilihan fitur
selected_feature_importance_df = feature_importance_df_xgb[feature_importance_df_xgb['Importance'] >= 0.030]

# Mengurutkan DataFrame berdasarkan importance secara descending
selected_feature_importance_df = selected_feature_importance_df.sort_values(by='Importance', ascending=False)

# Menampilkan bar chart
plt.figure(figsize=(6, 4))
sns.barplot(x='Importance', y='Feature', data=selected_feature_importance_df)
plt.title('Selected Feature Importance (XGBoost after Feature Selection)')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import datetime

# Fungsi untuk menghitung jumlah pelanggan yang dapat dihubungi per hari
def calculate_customers_per_day(call_duration, effective_hours, officers_count):
    return (effective_hours * 60) // call_duration * officers_count

# Fungsi untuk melakukan simulasi panggilan kampanye
def campaign_simulation(data, call_duration, officers_count, start_date, end_date, effective_hours):
    # Menghitung jumlah pelanggan yang dapat dihubungi per hari
    customers_per_day = calculate_customers_per_day(call_duration, effective_hours, officers_count)

    # Membuat kolom baru untuk tanggal panggilan
    data['call_date'] = np.random.choice(pd.date_range(start_date, end_date, freq='D'), len(data))

    # Menentukan pelanggan yang akan dihubungi setiap hari
    contacted_customers = data.groupby('call_date').apply(lambda x: x.sample(customers_per_day)).reset_index(drop=True)

    return contacted_customers[['call_date', 'age', 'job', 'marital', 'education', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']]

# Asumsi-asumsi
call_duration = 5  # Durasi panggilan dalam menit
effective_hours = 7  # Jam operasional bank dalam sehari
officers_count = 1  # Jumlah Marketing Officer
start_date = '2022-11-01'
end_date = '2022-12-16'

# Menjalankan simulasi panggilan kampanye
simulated_data = campaign_simulation(df, call_duration, officers_count, start_date, end_date, effective_hours)

# Menampilkan hasil simulasi
print(simulated_data.head())


In [ ]:
# Filtering data asli
filtered_original_data = simulated_data

# Filtering data hasil simulasi
filtered_simulated_data = simulated_data[
    (simulated_data['poutcome'] == 'success') &
    (simulated_data['duration'] > 240) &
    (simulated_data['housing'] == 'no') &
    (simulated_data['marital'].isin(['married', 'divorced']))
]

# Perbandingan jumlah data antara data asli dan hasil simulasi
print("Jumlah Data Asli setelah Filtering:", len(filtered_original_data))
print("Jumlah Data Hasil Simulasi setelah Filtering:", len(filtered_simulated_data))

# Menampilkan 5 data pertama dari kedua dataframe
print("\nData Asli setelah Filtering:")
print(filtered_original_data.head())

print("\nData Hasil Simulasi setelah Filtering:")
print(filtered_simulated_data.head())

In [ ]:
# Hitung persentase nilai y = yes pada Data Asli
percentage_original_data = (filtered_original_data['y'].value_counts(normalize=True) * 100).loc['yes']

# Hitung persentase nilai y = yes pada Data Hasil Simulasi
percentage_simulated_data = (filtered_simulated_data['y'].value_counts(normalize=True) * 100).loc['yes']

print(f"Persentase nilai y = 'yes' pada Data Asli: {percentage_original_data:.2f}%")
print(f"Persentase nilai y = 'yes' pada Data Hasil Simulasi: {percentage_simulated_data:.2f}%")

In [ ]:
simulated_data.describe()

In [ ]:
# Membuat kolom baru 'model_result' berdasarkan prediksi dari model XGBoost
X_test_combined_selected['model_result'] = best_model_xgb_selected.predict(X_test_combined_selected)

# Menampilkan beberapa baris pertama dari DataFrame dengan kolom 'model_result'
print(X_test_combined_selected[['model_result']].head())

# Membandingkan nilai "yes" dari model dengan "yes" pada data asli
comparison_df = pd.DataFrame({'Actual': y_test_binary_xgb, 'Model_Prediction': X_test_combined_selected['model_result']})

# Menampilkan beberapa baris pertama dari DataFrame perbandingan
print(comparison_df.head())

In [ ]:
# Menghitung perbandingan nilai "yes" per total populasi pada data asli
actual_yes_ratio = y_test_binary_xgb.sum() / len(y_test_binary_xgb)

# Menghitung perbandingan nilai "yes" per total populasi pada hasil model
model_yes_ratio = X_test_combined_selected['model_result'].sum() / len(X_test_combined_selected['model_result'])

# Menampilkan hasil perbandingan
print(f"Actual 'yes' ratio per total populasi: {actual_yes_ratio:.4f}")
print(f"Model 'yes' ratio per total populasi: {model_yes_ratio:.4f}")